In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.ticker as ticker

# 设置 Times New Roman 字体和全局字体大小
plt.rcParams['font.sans-serif'] = ['Times New Roman']  # 使用 Times New Roman 字体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号 '-' 显示为方块的问题
# plt.rcParams['axes.labelsize'] = 16      # 坐标轴标签字体大小
# plt.rcParams['xtick.labelsize'] = 14     # x轴刻度标签字体大小
# plt.rcParams['ytick.labelsize'] = 14     # y轴刻度标签字体大小
# plt.rcParams['legend.fontsize'] = 14     # 图例字体大小
# plt.rcParams['axes.titlesize'] = 18      # 标题字体大小

# 读取Excel文件
df = pd.read_excel('../data/steel_all.xlsx')

# 打印原始列名
print("原始列名:", df.columns.tolist())

# 丢弃"炉号"列（如果存在）
if '炉号' in df.columns:
    X = df.drop(columns=['炉号'])
    print("已丢弃'炉号'列")
else:
    X = df.copy()
    print("数据中不存在'炉号'列")

# 定义中英文列名映射字典
column_mapping = {
    'Fe': 'Fe',
    'SiO2': 'SiO2',
    'CaO': 'CaO',
    'MgO': 'MgO',
    'R(碱度)': 'R_alkalinity',
    'C': 'C',
    'Si': 'Si',
    'Mn': 'Mn',
    'P': 'P',
    'S': 'S',
    'T（温度k）': 'Temperature',
    '铁水量(吨)': 'Molten_iron',
    '废钢合计（吨）': 'Scrap_steel',
    '耗氧量合计（m3）': 'Oxygen_consumption',
    '石灰(Kg)': 'Lime',
    '石灰石(Kg)': 'Limestone',
    '白云石(Kg)': 'Dolomite',
    '镁球(Kg)': 'Magnesium_ball',
    '球团矿(Kg)': 'Pellet_ore',
    '返矿(Kg)': 'Return_ore',
    '烧结返矿(Kg)': 'Sintering_return_ore',
    '落地烧结矿(Kg)': 'Ground_sintering_ore',
    '块矿(Kg)': 'Lump_ore',
    '炉内固废球(Kg)': 'In_furnace_solid_waste_ball',
    '铁包固废球(Kg)': 'Iron_pot_solid_waste_ball',
    '固废球(Kg)': 'Solid_waste_ball',
    'N1': 'N1',
    'N2': 'N2',
    'N3': 'N3',
    'N4': 'N4',
    'n_total': 'n_total',
    'C_steel（%）': 'C_steel_pct',
    'Mn_steel（%）': 'Mn_steel_pct',
    'P_steel（%）': 'P_steel_pct',
    'S_steel（%）': 'S_steel_pct',
    '出钢温度(℃)': 'Tapping_temp_celsius'
}
# 重命名列
X_renamed = X.rename(columns=column_mapping)

# 计算每列缺失值的百分比
missing_info = pd.DataFrame({
    'Features': X_renamed.columns,
    'Missing_Percentage': (X_renamed.isnull().sum() / len(X_renamed)) * 100
})

# 按缺失率降序排序
missing_info_sorted = missing_info.sort_values('Missing_Percentage', ascending=False)

# 定义自定义分类函数
def categorize_missing(percentage):
    if percentage > 21:
        return 'Bad'
    elif percentage >= 5 and percentage <= 20:
        return 'Filling'
    elif percentage < 5:
        return 'OK'
    else:
        return 'Other'  # 35-60% 范围

# 应用分类
missing_info_sorted['Band'] = missing_info_sorted['Missing_Percentage'].apply(categorize_missing)

# 只保留有缺失值的特征
missing_info_sorted = missing_info_sorted[missing_info_sorted['Missing_Percentage'] > 0]

if not missing_info_sorted.empty:
    # 设置绘图样式
    plt.figure(figsize=(20, 12))
    sns.set(style="whitegrid")  # 移除font_scale参数，使用自定义字体大小
    
    # # 定义每个类别的颜色
    # color_palette = {
    #     'OK': '#00bec6',      # 青色 - 少于10%
    #     'Filling': '#ffa500',  # 橙色 - 10%到35%
    #     'Bad': '#f9766e',      # 红色 - 大于60%
    #     'Other': '#a9a9a9'     # 灰色 - 35%-60%
    # }
        # 定义每个类别的颜色（红黄绿经典配色）
    color_palette = {
        'OK': '#4CAF50',      # 绿色 - 少于10%
        'Filling': '#FFEB3B',  # 黄色 - 10%到35%
        'Bad': '#F44336',      # 红色 - 大于60%
        'Other': '#a9a9a9'     # 灰色 - 35%-60%（保持不变）
    }
    #     # 定义每个类别的颜色
    # color_palette = {
    #     'OK': (194/255, 229/255, 207/255),  # (194,229,207) 转换为RGB比例
    #     'Filling': (237/255, 221/255, 171/255),  # (237,221,171) 转换为RGB比例
    #     'Bad': (221/255, 115/255, 137/255),  # (221,115,137) 转换为RGB比例
    #     'Other': '#a9a9a9'  # 保持灰色不变
    # }
    
    # 绘制条形图
    barplot = sns.barplot(x='Missing_Percentage', y='Features', hue='Band', 
                        data=missing_info_sorted, dodge=False, 
                        palette=color_palette)
    
    # 设置背景和网格
    ax = plt.gca()
    ax.set_facecolor('whitesmoke')
    
    # 设置刻度标签字体大小
    ax.tick_params(axis='x', labelsize=25)  # x轴刻度标签
    ax.tick_params(axis='y', labelsize=25)  # y轴刻度标签
    
    # 修改刻度间隔为10
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))
    ax.grid(which='major', color='gray', linestyle='-', linewidth=0.5, axis='x')
    ax.grid(which='minor', color='gray', linestyle=':', linewidth=0.5, axis='x')
    
    # 调整x轴范围，为标签留出空间
    max_value = missing_info_sorted['Missing_Percentage'].max()
    plt.xlim(-0.25, (max_value // 10 + 1) * 10)  # 四舍五入到最接近的10的倍数
    
    # 在条形图上添加标签
    for i, row in missing_info_sorted.iterrows():
        band_color = color_palette[row.Band]
        barplot.text(row.Missing_Percentage, 
                   missing_info_sorted['Features'].tolist().index(row.Features), 
                   f"{row.Missing_Percentage:.2f}%", 
                   color='black', va="center", ha="left", fontsize=22,  # 设置标签字体大小
                   bbox=dict(facecolor='white', edgecolor='white', 
                           alpha=0.8, boxstyle='round,pad=0.2'))
    
                #        color='black', va="center", ha="left", fontsize=22,  # 设置标签字体大小
                #    bbox=dict(facecolor=band_color, edgecolor=band_color, 
                #            alpha=0.8, boxstyle='round,pad=0.2'))
    # 设置标题和标签(保持英文)
    plt.xlabel('Missing Percentage (%)', fontsize=22)
    plt.ylabel('Features', fontsize=22)
    plt.title('Distribution of Missing Rates for Each Feature', fontsize=24)
    
    # 添加图例说明(保持英文)
    legend_labels = {
        'OK': '<5% (Median filling)', 
        'Filling': '5%-20% (KNN filling)',
        'Bad': '>20% (Highly missing)',
        # 'Other': '35%-60% (Other)'
    }

    handles, labels = ax.get_legend_handles_labels()
    updated_labels = [legend_labels.get(label, label) for label in labels]
    # 设置图例位置向下移动
    plt.legend(handles=handles, labels=updated_labels, title='Missing Rate Categories', 
            loc='lower right', fontsize=20, title_fontsize=20, 
            bbox_to_anchor=(1.0, 0.6))  # 使用bbox_to_anchor控制图例位置
    

    plt.tight_layout()
    plt.show()
    
    # 计算数据集总体缺失情况
    total_cells = X_renamed.size
    total_missing = X_renamed.isnull().sum().sum()
    overall_missing_rate = (total_missing / total_cells) * 100
    
    print(f"\n数据集总体统计信息:")
    print(f"单元格总数: {total_cells}")
    print(f"缺失值总数: {total_missing}")
    print(f"总体缺失率: {overall_missing_rate:.2f}%")
    
    # 打印每个类别的特征数量
    band_counts = missing_info_sorted['Band'].value_counts().to_dict()
    print("\n每个类别的特征数量:")
    for band, count in band_counts.items():
        print(f"{legend_labels.get(band, band)}: {count} 个特征")
else:
    print("数据集中未发现缺失值。")

# 保存重命名的数据集
X_renamed.to_excel('steel_all_english.xlsx', index=False)